In [106]:
import pandas as pd
import numpy as np
import os
import re
from rapidfuzz import fuzz

## データフレームを日経コードに割り当てる

In [ ]:
extracted_df = pd.read_csv("./../data/tickers/src_csv/unique_jpx_ticker_name_pairs.csv")

In [61]:
print("len(extracted_df):", len(extracted_df))

len(extracted_df): 5383


In [62]:
del_list = []
print(f"Number of entries before deletion: {len(complete_df)}")
for ticker in complete_df['ticker']:
    if not re.match(r'^\d{4}$', str(ticker)):
        # print(f"Invalid ticker found: {ticker}")
        del_list.append(ticker)

deleted_df = complete_df[~complete_df['ticker'].isin(del_list)]
print(f"Number of entries after deletion: {len(deleted_df)}")
deleted_df.to_csv("./../data/tickers/cleaned_ticker_list.csv", index=False)

Number of entries before deletion: 7416
Number of entries after deletion: 7211


In [63]:
cleaned_df = pd.read_csv("./../data/tickers/cleaned_ticker_list.csv")
df_merged = pd.merge(extracted_df, cleaned_df, on=['ticker'], how='left')
df_merged.iloc[:, 1] = df_merged.iloc[:, 1].str.replace(' 普通株式', '', regex=False)
df_merged.to_csv("./../data/tickers/unique_jpx_ticker_name_code_triples_1.csv", index=False)

In [67]:
print("length of extracted:", len(df_merged))
print("length of cleaned:", len(cleaned_df))

length of extracted: 5383
length of cleaned: 7211


In [64]:
only_9999 = cleaned_df[cleaned_df['ticker'] == 9999]
only_9999.to_csv("./../data/tickers/tickers_with_only_9999.csv", index=False)

In [65]:
# df_mergedの'Name'とcleaned_dfの'index_jp'が一致したら、df_mergedの'nikkei'にcleaned_dfの'nikkei'を入れる
mask = df_merged['Name'] == df_merged['index_jp']
df_merged.loc[mask, 'nikkei'] = df_merged.loc[mask, 'nikkei']

In [66]:
df_merged.to_csv("./../data/tickers/unique_jpx_ticker_name_code_triples_2.csv", index=False)

## しっかり割り当てる！さっきまでのはお遊びさ！

### 0. 定数定義

In [136]:
PATH_TO_NIKKEI_STOCKS_DETAIL = "./../data/tickers/src_csv/nikkei_stocks_detail.csv"
PATH_TO_JPX_CODES_NAME_PAIRS = "./../data/tickers/src_csv/unique_jpx_ticker_name_pairs.csv"

### 1. maskで機械的にどれくらい埋められるか調べる

In [137]:
nikkei_df = pd.read_csv(PATH_TO_NIKKEI_STOCKS_DETAIL)
jpx_df = pd.read_csv(PATH_TO_JPX_CODES_NAME_PAIRS)

In [138]:
mask_jpx_1 = jpx_df['index_jp'].isin(nikkei_df['index_jp'])
mask_jpx_2 = jpx_df['ticker'].isin(nikkei_df['ticker'])

### 2. 日経側のDFから、割り当て可能なティッカーだけ抽出

In [139]:
nikkei_df = pd.read_csv(PATH_TO_NIKKEI_STOCKS_DETAIL)
jpx_df = pd.read_csv(PATH_TO_JPX_CODES_NAME_PAIRS)

In [140]:
mask_nikkei_1 = nikkei_df['index_jp'].isin(jpx_df['index_jp'])
mask_nikkei_2 = nikkei_df['ticker'].isin(jpx_df['ticker'])

### 3. 文字列同士の類似度によって、さらなる割り当てを目指す

In [166]:
mask_nikkei = mask_nikkei_1 | mask_nikkei_2
mask_jpx = mask_jpx_1 | mask_jpx_2
rest_nikkei_df = nikkei_df[~mask_nikkei]
rest_jpx_df = jpx_df[~mask_jpx]

In [168]:
def max_similarity(word):
    tmp = [(fuzz.ratio(word, target), target) for target in rest_nikkei_df['index_jp']]
    tmp.sort(key=lambda x: x[0], reverse=True)
    return tmp[0] if tmp else (0, None)

In [184]:
rest_jpx_df["index_jp"] = rest_jpx_df["index_jp"].str.replace(r'(・?ホールディング(ス)?)|ＨＤ|HD|グループ|コーポレーション|フィナンシャル', '', regex=True)
rest_jpx_df["index_jp_fuzz_max"], rest_jpx_df["index_jp_fuzz_max_target"] = zip(*rest_jpx_df["index_jp"].apply(max_similarity))

In [185]:
rest_jpx_df.head()

,ticker,index_jp,index_jp_fuzz_max,index_jp_fuzz_max_target
3942,7758,セコニック,100.000000,セコニック
1318,3819,インテック,100.000000,インテック
1075,3460,ジャパン・シニアリビング投資法人,96.774194,ジャパン・シニアリビング投資法
961,3285,野村不動産マスターファンド投資法人,93.750000,野村不動産マスターファンド投資
615,2712,スターバックス コーヒー ジャパン,93.750000,スターバックス コーヒー ジャ


In [198]:
mask_well_buzz = rest_jpx_df['index_jp_fuzz_max'] > 0

In [199]:
rest_jpx_df = rest_jpx_df.sort_values(by='index_jp_fuzz_max', ascending=False)

In [200]:
rest_jpx_df[mask_well_buzz].to_csv("./../data/tickers/extracted_over80%match_jpx_nikkei_tmp.csv", index=False)

### 4. 一致したリストをまとめる

In [225]:
masked_nikkei_df = nikkei_df[mask_nikkei_1]
merged_mask_nikkei_1 = jpx_df.merge(masked_nikkei_df, on=['index_jp'], how='inner', suffixes=('_jpx', '_nikkei'))
merged_mask_nikkei_1.rename(columns={'ticker_jpx': 'ticker'}, inplace=True)
merged_mask_nikkei_1.head()

,ticker,index_jp,ticker_nikkei,nikkei,ja,eng
0,1301,極洋,1301,N0000001,極洋,KYOKUYO
1,1352,ホウスイ,9999,N0000006,ホウスイ,HOHSUI
2,1375,雪国まいたけ,9999,N0016145,雪国まいたけ,YUKIGUNI MAITAKE
3,1376,カネコ種苗,1376,N0021804,カネコ種苗,KANEKO SEEDS
4,1378,雪国まいたけ,9999,N0016145,雪国まいたけ,YUKIGUNI MAITAKE


In [226]:
masked_nikkei_df = nikkei_df[mask_nikkei_2]
merged_mask_nikkei_2 = jpx_df.merge(masked_nikkei_df, on=['ticker'], how='inner', suffixes=('_jpx', '_nikkei'))
merged_mask_nikkei_2.head()

,ticker,index_jp_jpx,index_jp_nikkei,nikkei,ja,eng
0,1301,極洋,極洋,N0000001,極洋,KYOKUYO
1,1332,日本水産,ニッスイ,N0000003,ニッスイ,NISSUI
2,1333,マルハ,マルハニチロ,N0000004,マルハニチロ,MARUHA NICHIRO
3,1375,雪国まいたけ,ユキグニファクトリー,N0032599,ユキグニファクトリー,YUKIGUNI FACTORY
4,1376,カネコ種苗,カネコ種苗,N0021804,カネコ種苗,KANEKO SEEDS


In [227]:
manual_select_df = pd.read_csv("./../data/tickers/manual_adjust_tickers.csv")
merged_manual_select_df = manual_select_df.merge(nikkei_df, on=['index_jp'], how='inner', suffixes=('_jpx', '_nikkei'))
merged_manual_select_df.rename(columns={'ticker_jpx': 'ticker'}, inplace=True)
merged_manual_select_df.head()

,ticker,index_jp_tmp,index_jp_fuzz_max,index_jp,ticker_nikkei,nikkei,ja,eng
0,1334,マルハニチロ,85.714286,マルハニチロ食品,9999,N0000002,マルハニチロ食品,MARUHA NICHIRO FOODS
1,1351,宝幸水,80.000000,宝幸,9999,N0000005,宝幸,HOKO
2,1448,スペースバリュー,69.565217,スペースバリューホールディング,9999,N0032402,スペースバリューホールディング,SPACE VALUE HOLDINGS
3,1501,三井山,85.714286,三井鉱山,9999,N0000011,三井鉱山,MITSUI MINING
4,1601,帝石,66.666667,帝国石油,9999,N0000025,帝国石油,TEIKOKU OIL


In [228]:
tmp1_df, tmp2_df, tmp3_df = merged_mask_nikkei_1[['ticker', 'nikkei', 'ja', 'eng']], merged_mask_nikkei_2[['ticker', 'nikkei', 'ja', 'eng']], merged_manual_select_df[['ticker', 'nikkei', 'ja', 'eng']]
print("length of final_merged_df:", len(tmp1_df), len(tmp2_df), len(tmp3_df))

length of final_merged_df: 3337 3629 299


In [229]:
final_merged_df = pd.concat([tmp1_df, tmp2_df, tmp3_df]).drop_duplicates().reset_index(drop=True).sort_values(by='ticker')
print("length of final_merged_df:", len(final_merged_df))

length of final_merged_df: 4652


In [230]:
final_merged_df.head(10)

,ticker,nikkei,ja,eng
0,1301,N0000001,極洋,KYOKUYO
3337,1332,N0000003,ニッスイ,NISSUI
3338,1333,N0000004,マルハニチロ,MARUHA NICHIRO
4353,1334,N0000002,マルハニチロ食品,MARUHA NICHIRO FOODS
4354,1351,N0000005,宝幸,HOKO
1,1352,N0000006,ホウスイ,HOHSUI
2,1375,N0016145,雪国まいたけ,YUKIGUNI MAITAKE
3339,1375,N0032599,ユキグニファクトリー,YUKIGUNI FACTORY
3,1376,N0021804,カネコ種苗,KANEKO SEEDS
3340,1377,N0012004,サカタのタネ,SAKATA SEED


In [231]:
result = final_merged_df.set_index('nikkei')

In [232]:
result

,ticker,ja,eng
nikkei,,,
N0000001,1301,極洋,KYOKUYO
N0000003,1332,ニッスイ,NISSUI
N0000004,1333,マルハニチロ,MARUHA NICHIRO
N0000002,1334,マルハニチロ食品,MARUHA NICHIRO FOODS
N0000005,1351,宝幸,HOKO
...,...,...,...
N0016497,9993,ヤマザワ,YAMAZAWA
N0016178,9994,やまや,YAMAYA
N0002371,9995,グローセル,GLOSEL


In [233]:
result.to_csv("./../data/tickers/Tickers_with_Nikkei.csv")

### 5. 結果

- #### 割り当てできたティッカーと日経コードとの組み合わせ：4652（731削減した）

    - ##### 対応する日経コードがないティッカー（731）・重複していたティッカー（48：数の差からそう考えられる）が除外された

- #### 取得して利用可能な状態にある時系列データの数：5635